# Model Privacy Attack Lab — full sweep on a Colab GPU

This notebook runs every experiment in the lab and writes the results back
to your Google Drive so the repo's `results/` section can be populated.

**Before you start:** set the runtime to a GPU.
`Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`.
Without it, everything below will still run but will take roughly 20-40x
longer, which is the whole reason this notebook exists.

This notebook contains **no lab logic**. Every cell shells out to the
`experiments/` scripts, so the notebook can never drift from the library
code — that's the same rule the README states for notebooks.

Rough GPU timings (T4, defaults as committed):

| Step | Models trained | Approx. time |
|---|---|---|
| Targets (overfit + regularized) | 2 | ~5 min |
| MIA suite, per target | 25 | ~35 min |
| Gap sweep | 27 | ~30 min |
| Extraction | 10 | ~40 min |
| Defenses (incl. DP-SGD) | ~11 | ~60 min |

Budget around 3-4 hours for the whole thing. Colab free tier disconnects on
idle, so run it in chunks and keep the tab active, or run Section 7 to do
everything unattended.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n*** NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun. ***')

## 2. Get the code

Replace the URL with your own fork/repo. If you'd rather not push the repo
anywhere, zip the project folder, upload it via the Files pane, and
`!unzip -q Model-Security.zip` instead of cloning.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/Model-Security.git'  # <-- edit me
PROJECT_DIR = '/content/Model-Security'

import os
if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!ls

## 3. Install dependencies

Colab already ships a CUDA build of torch, so we install everything *except*
torch/torchvision — letting pip resolve those would trigger a slow reinstall
and can break the CUDA build.

In [ ]:
!pip install -q opacus pyyaml tqdm
import opacus, sklearn, matplotlib, scipy, pandas
print('opacus:', opacus.__version__, '| sklearn:', sklearn.__version__)

## 4. Sanity check before spending GPU hours

Run the test suite first. It takes about a minute and catches environment
problems (a bad opacus version, a CUDA mismatch) *before* you burn three
hours discovering them at the last step.

In [ ]:
!python -m pytest tests/ -q

## 5. Persist results to Drive (recommended)

Colab wipes `/content` when the runtime recycles. Symlinking `results/` into
Drive means a disconnect three hours in doesn't cost you the run.

Skip this cell if you'd rather download the artifacts manually at the end.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = '/content/drive/MyDrive/privacy-lab-results'
    !mkdir -p $DRIVE_RESULTS
    # Move any existing results aside rather than deleting them.
    !if [ -d results ] && [ ! -L results ]; then mv results results_local_backup; fi
    !ln -sfn $DRIVE_RESULTS results
    print('results/ ->', DRIVE_RESULTS)

## 6. Run the experiments

Each cell is independent and can be re-run on its own. Order matters only in
that the attacks need the targets to exist first.

### 6.1 Phase 1 — train the two targets (~5 min)

In [ ]:
!python experiments/train_targets.py --config configs/target_overfit.yaml
!python experiments/train_targets.py --config configs/target_regularized.yaml

### 6.2 Phases 2-4 — all MIA attacks (~35 min per target)

Trains 1 calibration model + 8 shadow models + 16 LiRA shadows per target.

Watch the **TPR@0.1%FPR** column rather than AUC — the headline result of
this lab is that the two can rank attacks differently.

In [ ]:
!python experiments/run_mia.py --config configs/mia_overfit.yaml

In [ ]:
!python experiments/run_mia.py --config configs/mia_regularized.yaml

### 6.3 Phase 2 experiment — gap vs leakage (~30 min)

27 models: 3 epoch settings x 3 training-set sizes x 3 seeds.

In [ ]:
!python experiments/run_gap_sweep.py --config configs/gap_sweep.yaml

### 6.4 Phase 5 — model extraction (~40 min)

Downloads CIFAR-100 (~169 MB) as the attacker's query pool on first run.

In [ ]:
!python experiments/run_extraction.py --config configs/extraction.yaml

### 6.5 Phase 6 — defenses and the DP-SGD sweep (~60 min)

The most expensive step, and the one that produces the core plot.
Add `--skip-extraction` to cut roughly half the time if you only need the
membership-inference columns.

In [ ]:
!python experiments/run_defenses.py --config configs/defenses.yaml

## 7. Unattended full run

Alternative to Section 6: runs everything in one go and logs to a file, so a
browser disconnect doesn't lose the output. Skip this if you ran 6.x.

In [ ]:
%%bash
set -e
exec > >(tee results/full_run.log) 2>&1
echo "=== started $(date) ==="
python experiments/train_targets.py --config configs/target_overfit.yaml
python experiments/train_targets.py --config configs/target_regularized.yaml
python experiments/run_mia.py        --config configs/mia_overfit.yaml
python experiments/run_mia.py        --config configs/mia_regularized.yaml
python experiments/run_gap_sweep.py  --config configs/gap_sweep.yaml
python experiments/run_extraction.py --config configs/extraction.yaml
python experiments/run_defenses.py   --config configs/defenses.yaml
echo "=== finished $(date) ==="

## 8. Review the results

In [ ]:
import pandas as pd
from pathlib import Path

for table in sorted(Path('results/tables').glob('*.csv')):
    print(f'\n=== {table.name} ===')
    print(pd.read_csv(table).to_string(index=False))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for plot in sorted(Path('results/plots').glob('*.png')):
    print(plot.name)
    display(Image(str(plot)))

## 9. Download the artifacts

Skip if you used the Drive symlink in Section 5 — they're already saved.

Checkpoints and `.npz` logit dumps are excluded: they're large, regenerable,
and `.gitignore`d anyway. What you want in the repo are the tables, plots,
and metrics JSON.

In [ ]:
!zip -r /content/lab_results.zip results/tables results/plots results/metrics
from google.colab import files
files.download('/content/lab_results.zip')

## 10. Next step

Unzip into the repo's `results/`, then fill in the README's **Results**
section with the real numbers and replace the *Key findings* hypotheses with
what actually happened — including anything that contradicted the
expectations. A finding that went against the hypothesis is more interesting
to a reader than one that confirmed it, and far more credible.

Then run the dashboard locally:

```bash
streamlit run app/streamlit_app.py
```